In [1]:
import os
from openai import OpenAI
import rich
import requests
import json

In [2]:
API_KEY = os.environ.get('AIHUBMIX_API_KEY')
BASE_URL = os.environ.get('AIHUBMIX_BASE_URL')
MODEL = "gpt-5-nano"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Calling a function and sending the result back to Chat API and Responses API**

Defining function that fetch current weather from API

In [3]:
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    print(f"get_weather function called to get weather for latitude = {latitude}, longitude = {longitude}")
    print(f"And result is  = {data['current']['temperature_2m']}")
    return data['current']['temperature_2m']

# Chat Completion API

https://platform.openai.com/docs/guides/function-calling?api-mode=chat

Defining the structure schema of the function to be passed as a tool in the API.

In [4]:
def get_weather_function_chat():
    return {
        "type": "function",
        "function": { # This property is removed from responses API
            "name": "get_weather",
            "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "Latitude of the location"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "Longitude of the location"
                    }
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False
            },
            "strict": True
        }
    }

First Step where model will responed with tool call request

In [5]:
messages=[
    {"role": "developer", "content": "You are a helpful assistant and provide update on weather in a city."},
    {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
]
tools = [get_weather_function_chat()]
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools = tools
)

rich.print(response.choices[0])
print("Finish Reason = ", response.choices[0].finish_reason)
rich.print(response.choices[0].message.tool_calls)


Choice(
    finish_reason='tool_calls',
    index=0,
    logprobs=None,
    message=ChatCompletionMessage(
        content=None,
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=[
            ChatCompletionMessageFunctionToolCall(
                id='call_9AHl0yymkE4AfuMVnqlbaMYa',
                function=Function(arguments='{"latitude": 24.86, "longitude": 67.01}', name='get_weather'),
                type='function',
                custom={}
            )
        ]
    )
)

Finish Reason =  tool_calls


[
    ChatCompletionMessageFunctionToolCall(
        id='call_9AHl0yymkE4AfuMVnqlbaMYa',
        function=Function(arguments='{"latitude": 24.86, "longitude": 67.01}', name='get_weather'),
        type='function',
        custom={}
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Chat API

To send the result of function call we need to send specific format object into Chat API call

```
new_message = {
    "role": "tool",
    "content": <output of function call>,
    "tool_call_id": tool_call.id
}
```

In [6]:
if response.choices[0].finish_reason == "tool_calls": # Check if finish_reason is tool_calls
    tool_call = response.choices[0].message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = { # Properties of this object will be different in responses API
        "role": "tool",
        "content": json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
        "tool_call_id": tool_call.id
    }
    # Important: we will append the previous message (response.choices[0].message)
    messages.append(response.choices[0].message)
    messages.append(new_message)
    # Calling the Chat API again with all the history messages and the new message
    response2 = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Model Response2 = ",response2.choices[0].message.content)
    # Notice the finish_reason of the response, it value is "stop"
    print("Finish Reason = ",response2.choices[0].finish_reason)

get_weather function called to get weather for latitude = 24.86, longitude = 67.01
And result is  = 22.9
Model Response2 =  Current in Karachi, Pakistan (approx 24.86°N, 67.01°E): temperature is about 22.9°C (73°F).

Would you like to know humidity, wind, or a forecast for the week?
Finish Reason =  stop


# Responses API

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

Note: Check the difference between json objects for function

In [7]:
def get_weather_function_response():
    return {
        "type": "function",  # There is no function property in the response API
        "name": "get_weather",
        "description": "Get the weather for a location. Call this whenever you need to know the weather, for example when a customer asks 'What's the weather like in this city'",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {
                    "type": "number",
                    "description": "Latitude of the location"
                },
                "longitude": {
                    "type": "number",
                    "description": "Longitude of the location"
                }
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }

### Using old way of sending history messages in every call

First Step where model will responed with tool call request

In [8]:
messages=[
    {"role": "developer", "content": "You are a helpful assistant and provide update on weather in a city."},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
     {"role": "user", "content": "Berlin"}
]
tools = [get_weather_function_response()]

response = openai.responses.create(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)
# rich.print(response)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_081eb8f7562541460069a6564d4ca881938ad7d22a54b30405',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ResponseFunctionToolCall(
        arguments='{"latitude":52.52,"longitude":13.405}',
        call_id='call_8pnDbLteMxQSqYHR73dcws09',
        name='get_weather',
        type='function_call',
        id='fc_081eb8f7562541460069a6564ee50881939743c83888d067fe',
        status='completed'
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

To send the result of function call we need to send specific format object into Responses API call

Note: object has different property names.

```
{
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": <output of function call>,
}
```

In [11]:
if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    arguments = json.loads(tool_call.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        # "output": str(weather)
        # Because of json object in output Responses API sometimes does not generate expected output
        "output":  json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
    }
    # Important: we will append the tool call (response.output[0]) and tool call ouput
    messages.append(response.output[-1])
    messages.append(new_message)
    # Calling the Responses API again with all the history messages and the new message
    response2 = openai.responses.create(model=MODEL, input=messages,tools = tools)
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)
    # rich.print(response2)

get_weather function called to get weather for latitude = 52.52, longitude = 13.405
And result is  = 4.0
Model Response2 =  Du meinst Berlin. Aktueller Standort: Berlin (52.52, 13.405). Wettercode: 4.0.

Ohne eine Legende dieses Codes kann ich dir nicht sicher sagen, was genau das bedeutet. Möchtest du, dass ich den Code in eine verständliche Beschreibung übersetze, sobald ich weiß, welche Zuordnung 4.0 hat, oder soll ich dir stattdessen eine normale Prognose (Temperatur, Niederschlag, Wind) geben, sobald diese verfügbar ist?
Status =  completed


### Using new way of conversation state by sending perivous reponse id

First Step where model will responed with tool call request

In [10]:
# This section is same as above

messages=[
    {"role": "developer", "content": "You are a helpful assistant and provide update on weather in a city."},
    # {"role": "user", "content": "What's the weather like in Karachi, Pakistan?"}
    # {"role": "user", "content": "NYC"}
     {"role": "user", "content": "Berline"}
]
tools = [get_weather_function_response()]

response = openai.responses.create(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_01ee570853a2702e0069a659f8beb881968a810853b28c2ff8',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ResponseFunctionToolCall(
        arguments='{"latitude":52.52,"longitude":13.405}',
        call_id='call_7a1zQ6A3jMHRYhBhoT792ZlV',
        name='get_weather',
        type='function_call',
        id='fc_01ee570853a2702e0069a659fa99b8819698b8c784458f0501',
        status='completed'
    )
]

Second Step where we are calling the `get_weather` function and sending the response back to Responses API

The only difference in below section is how messages are sent.

In [12]:
if response.output[-1].type == "function_call": # Check if output type is function_call
    tool_call = response.output[-1]
    arguments = json.loads(tool_call.arguments)
    latitude = arguments.get("latitude")
    longitude = arguments.get("longitude")
    # weather = get_weather(latitude, longitude) # Both will work
    weather = get_weather(**arguments)
    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        # "output": str(weather)
        # Because of json object in output Responses API sometimes does not generate expected output
        "output":  json.dumps({"latitude": latitude, "longitude": longitude, "weather": weather}),
    }
    # Not needed now, because we are sending the previous response id
    # messages.append(response.output[0])

    # Emptying the messages array because we are sending the previous response id,
    # therefore we don't need to send the previous message
    messages = []
    messages.append(new_message)
    # Note the previous_response_id
    response2 = openai.responses.create(model=MODEL, input=messages,tools = tools, previous_response_id=response.id)
    print("Model Response2 = ",response2.output_text)
    print("Status = ",response2.status)

get_weather function called to get weather for latitude = 52.52, longitude = 13.405
And result is  = 4.0
Model Response2 =  Did you mean Berlin, Germany? I can pull the current weather for you.

Right now I have Berlin’s weather coded as 4.0 from my feed, but I don’t have a plain-language description for that code. Would you like me to:
- fetch a more detailed, plain-language update (temperature, conditions, wind, etc.), or
- provide a 7-day forecast?

Also tell me your preferred units (Celsius or Fahrenheit).
Status =  completed
